# Lakebase Search execution — Build 2 assist retrieval
Proves the app's assist retrieves from the **Build 1 Lakebase Search index** (`ops.rm_notes_tsv_bm25`, BM25 full-text over `ops.rm_notes`), not a separate vector store.
This is the same query the app runs in `retrieveNotes()` before drafting a memo.

In [1]:
import os, subprocess, json
os.environ['PATH'] = '/opt/homebrew/opt/postgresql@16/bin:' + os.environ.get('PATH', '')
PROFILE = 'reyden-whisperers'
BR = 'projects/meridian-bank/branches/production'
EP = BR + '/endpoints/primary'
def cli(args):
    return json.loads(subprocess.run(['databricks', *args, '-p', PROFILE, '-o', 'json'],
                                      capture_output=True, text=True).stdout)
host = cli(['postgres', 'list-endpoints', BR])[0]['status']['hosts']['host']
email = cli(['current-user', 'me'])['userName']
token = cli(['postgres', 'generate-database-credential', EP])['token']
conn = f'host={host} port=5432 dbname=databricks_postgres user={email} sslmode=require'
print('Connected to Lakebase endpoint:', host)

Connected to Lakebase endpoint: ep-lingering-voice-d86p80aa.database.us-east-2.cloud.databricks.com


## Run the hybrid Lakebase Search — BM25 branch
Ranked full-text retrieval over `ops.rm_notes`, scoped to a customer (more-negative `bm25_score` = better match).

In [2]:
QUERY = 'retention balance rate offer maturity outflow'
CUST = 'CUST-0001955'
sql = (
    'SELECT note_id, author, left(note_text,110) AS note_text, '
    "(note_tsv <@> to_bm25query(to_tsvector('english', $q$" + QUERY + "$q$), "
    "'ops.rm_notes_tsv_bm25'::regclass)) AS bm25_score "
    'FROM ops.rm_notes WHERE customer_id = $c$' + CUST + '$c$ '
    'ORDER BY bm25_score ASC LIMIT 5;'
)
res = subprocess.run(['psql', conn, '-P', 'pager=off', '-c', sql],
                     capture_output=True, text=True, env=dict(os.environ, PGPASSWORD=token))
print(res.stdout or res.stderr)

 note_id | author  |                                                   note_text                                                    |     bm25_score      
---------+---------+----------------------------------------------------------------------------------------------------------------+---------------------
       1 | a.silva | Customer called about a maturing 12-month CD. Rate-sensitive and actively comparing competitor savings rates o | -3.8928382640434815
       4 | rm.desk | Left voicemail offering a promotional CD renewal rate. Awaiting response.                                      | -1.3585933375865242
(2 rows)


